# ADS-B I/Q transmitter ID classification with K-fold validation

Adapting the earlier raw I/Q classifier towards the ADS-B direction we have been discussing. The goal here is not general modulation classification anymore. I am trying to treat the ADS-B bursts as physical-layer I/Q captures and classify which aircraft/transponder they came from.

**Science Data Bank ADS-B real-world data set (DF=17) https://www.scidb.cn/en/detail?dataSetId=beea96eecdf949a086c4483503c4b289**.

The rough workflow here is: load complex I/Q data, split likely ADS-B bursts into fixed-size windows, normalise them, train a small 1D CNN, and check performance with stratified K-fold validation


In [ ]:
# Optional setup
# Oly need this if the environment is missing a package.
# %pip install numpy pandas scipy scikit-learn matplotlib torch tqdm


In [ ]:
# Configuration
from pathlib import Path

CONFIG = {
    # Local folder where I have extracted the DF17 I/Q files.
    # I have assumed one capture file per aircraft/transmitter.
    "dataset_root": Path("./adsb_iq"),

    # ADS-B / Mode S assumptions
    "sample_rate_hz": 40_000_000,      # dataset listing says 40 MHz
    "adsb_burst_us": 120,              # 8 us preamble + 112 us payload
    "burst_padding_us": 10,            # small amount of context either side

    # CNN windows
    # At 40 MHz, 120 us is 4800 complex samples. I am starting with 4096.
    # This should be a bit more manageable for early testing and later hardware work.
    "window_len": 4096,
    "stride": 2048,
    "normalise_per_window": True,
    "demean": True,

    # Simple burst picking
    "energy_ma_samples": 32,
    "threshold_mad_multiplier": 8.0,
    "min_burst_gap_samples": 800,
    "min_burst_len_samples": 160,
    "max_bursts_per_file": 400,

    # Avoid classes with hardly any usable windows
    "min_windows_per_class": 10,
    "max_windows_per_class": 300,

    # Cross-validation
    "n_splits": 5,
    "random_seed": 42,

    # Training
    "batch_size": 128,
    "epochs": 15,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,

    # Model
    "dropout_conv": 0.20,
    "dropout_fc": 0.50,
    "num_filters_1": 32,
    "num_filters_2": 64,
    "num_filters_3": 96,
    "kernel_size": 7,
    "fc_dim": 256,

    # Outputs
    "artifact_dir": Path("./artifacts_adsb_fingerprint"),
}

CONFIG["artifact_dir"].mkdir(parents=True, exist_ok=True)
CONFIG


## Data setup

I have set this up so the notebook can point at the extracted Science Data Bank DF17 folder. The loader is intentionally a bit flexible because I am not assuming the exact file format will always be the same after download/extraction.

It should handle:

- `.npy` or `.npz` arrays
- `.mat` files
- `.csv` or `.txt` files with I/Q columns
- binary `.bin`, `.dat`, `.iq`, or `.sigmf-data` captures

For binary files I may need to change `BINARY_IQ_FORMAT`. I have left it as `float32_iq` for now, but SDR captures are often saved as interleaved I/Q in a few different formats.


In [ ]:
# Imports and reproducibility
import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.ndimage import uniform_filter1d

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["random_seed"])


In [ ]:
# Raw I/Q file loading
# I may need to change this after checking the downloaded files.
# Options: "complex64", "float32_iq", "int16_iq", "uint8_iq", "int8_iq"
BINARY_IQ_FORMAT = "float32_iq"

SUPPORTED_SUFFIXES = {".npy", ".npz", ".mat", ".csv", ".txt", ".bin", ".dat", ".iq", ".sigmf-data"}


def _as_complex_vector(arr) -> np.ndarray:
    """Convert the usual I/Q layouts into one complex vector."""
    arr = np.asarray(arr)
    arr = np.squeeze(arr)

    if arr.size == 0:
        raise ValueError("Empty array")

    if np.iscomplexobj(arr):
        return arr.astype(np.complex64).reshape(-1)

    # Handles I/Q columns, transposed I/Q, or flat interleaved samples.
    if arr.ndim == 2 and arr.shape[-1] == 2:
        return (arr[:, 0] + 1j * arr[:, 1]).astype(np.complex64)
    if arr.ndim == 2 and arr.shape[0] == 2:
        return (arr[0, :] + 1j * arr[1, :]).astype(np.complex64)
    if arr.ndim == 1 and arr.size % 2 == 0:
        arr = arr.astype(np.float32)
        return (arr[0::2] + 1j * arr[1::2]).astype(np.complex64)

    raise ValueError(f"Cannot interpret array as I/Q, shape={arr.shape}, dtype={arr.dtype}")


def load_binary_iq(path: Path, fmt: str = BINARY_IQ_FORMAT) -> np.ndarray:
    if fmt == "complex64":
        return np.fromfile(path, dtype=np.complex64)
    if fmt == "float32_iq":
        raw = np.fromfile(path, dtype=np.float32)
        return _as_complex_vector(raw)
    if fmt == "int16_iq":
        raw = np.fromfile(path, dtype=np.int16).astype(np.float32) / 32768.0
        return _as_complex_vector(raw)
    if fmt == "int8_iq":
        raw = np.fromfile(path, dtype=np.int8).astype(np.float32) / 128.0
        return _as_complex_vector(raw)
    if fmt == "uint8_iq":
        raw = (np.fromfile(path, dtype=np.uint8).astype(np.float32) - 127.5) / 127.5
        return _as_complex_vector(raw)
    raise ValueError(f"Unknown binary format: {fmt}")


def load_iq_file(path: Path) -> np.ndarray:
    suffix = path.suffix.lower()

    if suffix == ".npy":
        return _as_complex_vector(np.load(path, allow_pickle=False))

    if suffix == ".npz":
        data = np.load(path, allow_pickle=False)
        # Try the obvious array names first.
        for key in ["iq", "x", "data", "samples", "signal"]:
            if key in data.files:
                return _as_complex_vector(data[key])
        return _as_complex_vector(data[data.files[0]])

    if suffix == ".mat":
        mat = loadmat(path)
        candidates = []
        for key, value in mat.items():
            if key.startswith("__"):
                continue
            arr = np.asarray(value)
            if arr.size >= CONFIG["window_len"]:
                candidates.append((key, arr))
        if not candidates:
            raise ValueError(f"No usable arrays found in {path}")
        # Use the largest useful array if there are a few options.
        key, arr = max(candidates, key=lambda kv: np.asarray(kv[1]).size)
        return _as_complex_vector(arr)

    if suffix in {".csv", ".txt"}:
        try:
            df = pd.read_csv(path)
        except Exception:
            df = pd.read_csv(path, header=None, delim_whitespace=True)

        lower_cols = [str(c).lower() for c in df.columns]
        if "i" in lower_cols and "q" in lower_cols:
            i_col = df.columns[lower_cols.index("i")]
            q_col = df.columns[lower_cols.index("q")]
            return (df[i_col].to_numpy() + 1j * df[q_col].to_numpy()).astype(np.complex64)
        if df.shape[1] >= 2:
            return (df.iloc[:, 0].to_numpy() + 1j * df.iloc[:, 1].to_numpy()).astype(np.complex64)
        # Last fallback for text files saved as complex values.
        return np.loadtxt(path, dtype=np.complex64).reshape(-1)

    if suffix in {".bin", ".dat", ".iq", ".sigmf-data"}:
        return load_binary_iq(path)

    raise ValueError(f"Unsupported file type: {path}")


def infer_label_from_path(path: Path) -> str:
    """Use the filename as the aircraft/transmitter label."""
    stem = path.stem
    # Strip capture-format words from the label where possible.
    stem = re.sub(r"(_?iq|_?df17|_?adsb|_?40mhz|_?capture|_?samples)$", "", stem, flags=re.I)
    return stem

In [ ]:
# ADS-B burst detection and windowing

def robust_energy_threshold(power: np.ndarray, mad_multiplier: float) -> float:
    med = np.median(power)
    mad = np.median(np.abs(power - med)) + 1e-12
    return med + mad_multiplier * 1.4826 * mad


def find_energy_regions(mask: np.ndarray, min_gap: int, min_len: int):
    """Join nearby regions and drop tiny ones."""
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return []

    regions = []
    start = prev = int(idx[0])
    for i in idx[1:]:
        i = int(i)
        if i - prev <= min_gap:
            prev = i
        else:
            if prev - start + 1 >= min_len:
                regions.append((start, prev + 1))
            start = prev = i
    if prev - start + 1 >= min_len:
        regions.append((start, prev + 1))
    return regions


def extract_adsb_bursts(iq: np.ndarray, config=CONFIG):
    """Pull out likely ADS-B burst snippets from a longer I/Q capture.

    This is a simple first pass based on energy. A proper preamble/DF17 detector should replace this later.
    """
    iq = np.asarray(iq, dtype=np.complex64).reshape(-1)
    power = np.abs(iq) ** 2
    smooth = uniform_filter1d(power, size=config["energy_ma_samples"], mode="nearest")
    threshold = robust_energy_threshold(smooth, config["threshold_mad_multiplier"])
    mask = smooth > threshold

    regions = find_energy_regions(
        mask,
        min_gap=config["min_burst_gap_samples"],
        min_len=config["min_burst_len_samples"],
    )

    burst_len = int((config["adsb_burst_us"] + 2 * config["burst_padding_us"]) * 1e-6 * config["sample_rate_hz"])
    pre = int(config["burst_padding_us"] * 1e-6 * config["sample_rate_hz"])

    bursts = []
    for start, end in regions[: config["max_bursts_per_file"]]:
        centre = (start + end) // 2
        b0 = max(0, centre - burst_len // 2)
        b1 = min(len(iq), b0 + burst_len)
        b0 = max(0, b1 - burst_len)
        snippet = iq[b0:b1]
        if len(snippet) >= config["window_len"]:
            bursts.append(snippet)
    return bursts, {"threshold": float(threshold), "num_regions": len(regions), "burst_len": burst_len, "pre": pre}


def normalise_window(w: np.ndarray, config=CONFIG) -> np.ndarray:
    w = np.asarray(w, dtype=np.complex64)
    if config["demean"]:
        w = w - np.mean(w)
    if config["normalise_per_window"]:
        rms = np.sqrt(np.mean(np.abs(w) ** 2)) + 1e-12
        w = w / rms
    return w


def iq_to_channels(w: np.ndarray) -> np.ndarray:
    return np.stack([w.real, w.imag], axis=0).astype(np.float32)


def window_burst(burst: np.ndarray, config=CONFIG):
    L = config["window_len"]
    stride = config["stride"]
    if len(burst) < L:
        return []
    windows = []
    for start in range(0, len(burst) - L + 1, stride):
        w = normalise_window(burst[start:start + L], config)
        windows.append(iq_to_channels(w))
    return windows

In [ ]:
# Build the ADS-B transmitter-ID dataset

def find_iq_files(root: Path):
    root = Path(root)
    return sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_SUFFIXES])


def build_adsb_window_dataset(root: Path, config=CONFIG):
    files = find_iq_files(root)
    if not files:
        raise FileNotFoundError(
            f"No supported I/Q files found under {root.resolve()}. "
            "Download/extract the Science Data Bank DF17 dataset and update CONFIG['dataset_root']."
        )

    rows = []
    file_summaries = []

    for path in files:
        label = infer_label_from_path(path)
        try:
            iq = load_iq_file(path)
            bursts, info = extract_adsb_bursts(iq, config)
            n_windows_before = len(rows)
            for burst_idx, burst in enumerate(bursts):
                for win_idx, x in enumerate(window_burst(burst, config)):
                    rows.append({
                        "x": x,
                        "y": label,
                        "file": str(path),
                        "group": f"{path.name}::burst{burst_idx}",
                        "burst_idx": burst_idx,
                        "win_idx": win_idx,
                    })
            file_summaries.append({
                "file": str(path),
                "label": label,
                "iq_samples": len(iq),
                "bursts": len(bursts),
                "windows": len(rows) - n_windows_before,
                **info,
            })
        except Exception as e:
            file_summaries.append({
                "file": str(path),
                "label": label,
                "error": repr(e),
                "bursts": 0,
                "windows": 0,
            })

    # Drop tiny classes, then cap each class so one aircraft does not dominate.
    counts = Counter(r["y"] for r in rows)
    keep_classes = {cls for cls, n in counts.items() if n >= config["min_windows_per_class"]}
    rows = [r for r in rows if r["y"] in keep_classes]

    if config["max_windows_per_class"] is not None:
        capped = []
        per_class = defaultdict(int)
        rng = np.random.default_rng(config["random_seed"])
        order = rng.permutation(len(rows))
        for idx in order:
            r = rows[int(idx)]
            if per_class[r["y"]] < config["max_windows_per_class"]:
                capped.append(r)
                per_class[r["y"]] += 1
        rows = capped

    return rows, pd.DataFrame(file_summaries)


rows, file_summary = build_adsb_window_dataset(CONFIG["dataset_root"])

print("Files inspected:", len(file_summary))
print("Classes kept:", len(set(r["y"] for r in rows)))
print("Total windows:", len(rows))
print("Example class counts:", Counter(r["y"] for r in rows).most_common(10))

file_summary.head()

In [ ]:
# Encode labels and have a quick look at the data
X = np.stack([r["x"] for r in rows])
y_text = np.array([r["y"] for r in rows])
groups = np.array([r["group"] for r in rows])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)
class_names = list(label_encoder.classes_)

print("X shape:", X.shape)          # windows, I/Q channels, samples
print("y shape:", y.shape)
print("Number of classes:", len(class_names))

class_table = pd.DataFrame({
    "class": class_names,
    "encoded": np.arange(len(class_names)),
    "windows": [np.sum(y == i) for i in range(len(class_names))],
})
class_table.sort_values("windows", ascending=False).head(20)

In [ ]:
# Quick sanity plots

def plot_examples(X, y, class_names, max_classes=4):
    chosen = []
    for class_idx in range(min(len(class_names), max_classes)):
        indices = np.where(y == class_idx)[0]
        if len(indices):
            chosen.append(indices[0])

    for idx in chosen:
        plt.figure(figsize=(12, 3))
        plt.plot(X[idx, 0], label="I")
        plt.plot(X[idx, 1], label="Q")
        plt.title(f"Example ADS-B I/Q window - class {class_names[y[idx]]}")
        plt.xlabel("Sample")
        plt.ylabel("Normalised amplitude")
        plt.legend()
        plt.tight_layout()
        plt.show()

plot_examples(X, y, class_names)

## Model

For this baseline I am using a compact 1D CNN on raw I/Q windows. I am deliberately not feeding decoded ADS-B fields into the model, because that would make the problem too easy and not really test the transmitter fingerprinting idea.

The main thing I want from this stage is a software baseline that we can later compare against a smaller/quantised version for RFSoC work \cite{soltani2019realtimeembeddeddeeplearning}.


In [ ]:
# PyTorch dataset and model
class IQWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class ADSBIQCNN(nn.Module):
    def __init__(self, num_classes: int, window_len: int):
        super().__init__()
        k = CONFIG["kernel_size"]
        pad = k // 2
        self.features = nn.Sequential(
            nn.Conv1d(2, CONFIG["num_filters_1"], kernel_size=k, padding=pad),
            nn.BatchNorm1d(CONFIG["num_filters_1"]),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CONFIG["dropout_conv"]),

            nn.Conv1d(CONFIG["num_filters_1"], CONFIG["num_filters_2"], kernel_size=k, padding=pad),
            nn.BatchNorm1d(CONFIG["num_filters_2"]),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(CONFIG["dropout_conv"]),

            nn.Conv1d(CONFIG["num_filters_2"], CONFIG["num_filters_3"], kernel_size=k, padding=pad),
            nn.BatchNorm1d(CONFIG["num_filters_3"]),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(32),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(CONFIG["num_filters_3"] * 32, CONFIG["fc_dim"]),
            nn.ReLU(),
            nn.Dropout(CONFIG["dropout_fc"]),
            nn.Linear(CONFIG["fc_dim"], num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# Training and evaluation helpers
def make_loaders(train_idx, val_idx):
    train_ds = IQWindowDataset(X[train_idx], y[train_idx])
    val_ds = IQWindowDataset(X[val_idx], y[val_idx])
    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
    return train_loader, val_loader


def train_one_epoch(model, loader, criterion, optimiser):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimiser.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds = []
    targets = []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(yb)
        preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        targets.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(targets, preds),
        "macro_f1": f1_score(targets, preds, average="macro"),
        "preds": preds,
        "targets": targets,
    }

In [ ]:
# Stratified K-fold cross-validation
# If each aircraft only has one file, I cannot really split by file yet.
# This is therefore window-level validation under the current capture conditions.
# A stronger test would need multiple days, receivers, or passes per aircraft.

skf = StratifiedKFold(
    n_splits=CONFIG["n_splits"],
    shuffle=True,
    random_state=CONFIG["random_seed"],
)

fold_results = []
all_fold_preds = []
all_fold_targets = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== Fold {fold}/{CONFIG['n_splits']} =====")
    set_seed(CONFIG["random_seed"] + fold)

    train_loader, val_loader = make_loaders(train_idx, val_idx)
    model = ADSBIQCNN(num_classes=len(class_names), window_len=CONFIG["window_len"]).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    best_state = None
    best_macro_f1 = -1.0
    history = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimiser)
        val_metrics = evaluate(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": train_loss, **{k: v for k, v in val_metrics.items() if k not in ["preds", "targets"]}})

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"acc={val_metrics['accuracy']:.4f} | macro_f1={val_metrics['macro_f1']:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics = evaluate(model, val_loader, criterion)

    fold_results.append({
        "fold": fold,
        "accuracy": final_metrics["accuracy"],
        "macro_f1": final_metrics["macro_f1"],
        "loss": final_metrics["loss"],
    })
    all_fold_preds.append(final_metrics["preds"])
    all_fold_targets.append(final_metrics["targets"])

    torch.save(model.state_dict(), CONFIG["artifact_dir"] / f"adsb_iqcnn_fold{fold}.pt")
    pd.DataFrame(history).to_csv(CONFIG["artifact_dir"] / f"history_fold{fold}.csv", index=False)

results_df = pd.DataFrame(fold_results)
results_df

In [ ]:
# Summarise results
print(results_df.describe())

all_preds = np.concatenate(all_fold_preds)
all_targets = np.concatenate(all_fold_targets)

print("\nOverall classification report:")
print(classification_report(all_targets, all_preds, target_names=class_names, zero_division=0))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
plt.imshow(cm, aspect="auto")
plt.title("ADS-B transmitter ID confusion matrix")
plt.xlabel("Predicted transmitter")
plt.ylabel("True transmitter")
plt.colorbar(label="Count")
plt.tight_layout()
plt.show()

results_df.to_csv(CONFIG["artifact_dir"] / "kfold_results.csv", index=False)
with open(CONFIG["artifact_dir"] / "label_mapping.json", "w") as f:
    json.dump({int(i): name for i, name in enumerate(class_names)}, f, indent=2)

print("Saved artefacts to:", CONFIG["artifact_dir"].resolve())

## Notes for next steps

This notebook is still just a baseline for the offline experiments. It is not the final real-time RFSoC version.

The next things to look at are:

1. Replacing the energy-only burst extraction with a proper ADS-B preamble detector so the packet alignment is cleaner.
2. Keeping decoded identifiers out of the model inputs. They are useful for labelling the data, but not for proving a physical-layer fingerprinting approach.
3. Collecting more variation for the same aircraft/transponders, ideally across different days, passes, receiver positions, and SNRs. Otherwise the model could learn channel/session effects instead of transmitter effects.
4. Testing an open-set version where the model can reject an unknown transmitter rather than forcing every example into a known class.
5. Once the offline results look stable, shrinking the model for the ZCU111/RFSoC direction using shorter windows, quantisation-aware training, pruning, or some lightweight feature extraction before the neural network
